In [1]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [2]:
ORDER = {0: 'First', 1: 'Second', 2: 'Third', 3: 'Total'}

In [3]:
def DataParser(minute):
    path = f'./Data/{minute}m break/'
    filelist = os.listdir(path)
    ids = set([file[:2] for file in filelist])
    group = {}

    for idx in ids:
        data = []
        for i in range(3):
            data.append(pd.read_csv(f'./Data/{minute}m break/{idx}_{i+1}.csv', index_col=0).T.reset_index())
            data[i]['index'] = data[i]['index'].astype(int)
            data[i]['class'] = data[i]['class'].astype(int)
            data[i]['speed'] = data[i]['speed'].astype(float)
        data.append(data[0])
        data[3] = pd.concat([data[3], data[1], data[2]], axis=0, ignore_index=True)
        data[3] = data[3].iloc[:, 1:].reset_index()
        group[idx] = data
    
    return group

In [4]:
Group = {}
for i in [0, 5, 10]:
    Group[i] = DataParser(i)

# 1
### 전체 데이터에 대한 반응속도 추세선의 기울기 간의 상대 점수

#### a. 전체에 대해

In [25]:
def all_gradient_z_score():
    gradient = {}
    for g in Group.values():
        for k, p in g.items():
            speed = p[3]['speed']
    
            x = np.arange(len(speed))
            y = speed
    
            coeffs = np.polyfit(x, y, 1)

            slope = coeffs[0]
            gradient[k] = slope
    gradient_np = np.array(list(gradient.values()))

    mean_slope = np.mean(gradient_np)
    std_slope = np.std(gradient_np)
    z_score = (gradient_np - mean_slope) / std_slope
    
    zperid = {}
    for idx, score in zip(gradient.keys(), z_score):
        zperid[idx] = round(score, 3)

    sorted_zscore = {k:v for k, v in sorted(zperid.items())}

    df = pd.DataFrame(list(sorted_zscore.items()), columns = ['id', 'z_score'])
    df['rank'] = df['z_score'].rank(ascending=False, method='min').astype(int)
    df.set_index('id', inplace=True)
    df.to_excel('./Graphs/z_score_all.xlsx')
    return df

In [27]:
all_gradient_z_score()

,z_score,rank
id,,
01,0.159,12
02,0.813,6
03,0.660,7
04,1.352,2
05,-0.503,20
06,1.118,4
07,-2.049,28
08,-0.224,16
09,0.300,11


#### b. 그룹 별

In [28]:
def Group_gradient_z_score():
    for m, g in Group.items():
        gradient = {}
        for k, p in g.items():
            speed = p[3]['speed']
    
            x = np.arange(len(speed))
            y = speed
    
            coeffs = np.polyfit(x, y, 1)

            slope = coeffs[0]
            gradient[k] = slope
        gradient_np = np.array(list(gradient.values()))
    
        mean_slope = np.mean(gradient_np)
        std_slope = np.std(gradient_np)
        z_score = (gradient_np - mean_slope) / std_slope
        
        zperid = {}
        for idx, score in zip(gradient.keys(), z_score):
            zperid[idx] = round(score, 3)
    
        sorted_zscore = {k:v for k, v in sorted(zperid.items())}
    
        df = pd.DataFrame(list(sorted_zscore.items()), columns = ['id', 'z_score'])
        df['rank'] = df['z_score'].rank(ascending=False, method='min').astype(int)
        df.set_index('id', inplace=True)
        df.to_excel(f'./Graphs/z_score_group_{m}.xlsx')

In [29]:
Group_gradient_z_score()

# 2
### 그룹 3개에 대한 블록 3개의 데이터 표
평균, 분산

In [34]:
speed = {}
for m, g in Group.items():
    speed[m] = {'block 1': [], 'block 2': [], 'block 3': []}
    for k, p in g.items():
        for i in range(3):
            speed[m][f'block {i+1}'].append(p[i]['speed'])

data_m = {0:[], 5:[], 10:[]}
data_v = {0:[], 5:[], 10:[]}
for m, g in speed.items():
    for block, lists in g.items():
        block_all = []
        for p in lists:
            for s in p:
                block_all.append(s)
        mean = np.mean(block_all)
        var = np.var(block_all)

        data_m[m].append(mean)
        data_v[m].append(var)

In [43]:
df_m = pd.DataFrame(data_m)
df_m.index=['Block 1', 'Block 2', 'Block 3']
df_m.to_excel('./Graphs/Mean_of_data.xlsx')
df_m

,0,5,10
Block 1,0.775976,0.670519,0.880121
Block 2,0.795824,0.632092,0.858837
Block 3,0.873295,0.641781,0.912771


In [44]:
df_v = pd.DataFrame(data_v)
df_v.index=['Block 1', 'Block 2', 'Block 3']
df_v.to_excel('./Graphs/Varience_of_data.xlsx')
df_v

,0,5,10
Block 1,0.157558,0.143865,0.332692
Block 2,0.149964,0.079481,0.279438
Block 3,0.205515,0.082976,0.274244
